In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_blobs, make_moons, load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

np.random.seed(42)
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 30)
print("Библиотеки загружены. Версия pandas:", pd.__version__)

---
# Пара 1. Снижение размерности: SVD и PCA

## 1.1. Зачем уменьшать число признаков

Если таблица содержит 100 признаков, это не означает, что в данных действительно есть 100 независимых направлений информации. Многие столбцы коррелируют и частично повторяют друг друга.

Снижение размерности пытается представить объект меньшим числом новых признаков, сохранив как можно больше полезной структуры.

Типичные цели:

- визуализация многомерных данных в 2D или 3D;
- ускорение обучения модели;
- уменьшение шума и мультиколлинеарности;
- сжатие данных;
- подготовка признаков перед классификацией или кластеризацией.

Важно: новые компоненты PCA обычно хуже интерпретируются, чем исходные признаки.

## 1.2. SVD на маленькой матрице

SVD раскладывает матрицу $X$ на три матрицы:

$$X = U \Sigma V^T$$

- $U$ описывает направления объектов;
- $\Sigma$ хранит сингулярные значения — «силу» направлений;
- $V^T$ описывает направления признаков.

Если оставить только несколько самых больших сингулярных значений, получим приближение исходной матрицы меньшего ранга.

In [ ]:
X_small = np.array([
    [5, 5, 0, 0],
    [4, 4, 0, 0],
    [0, 0, 5, 5],
    [0, 0, 4, 4],
], dtype=float)

U, singular_values, VT = np.linalg.svd(X_small, full_matrices=False)
print("Сингулярные значения:", np.round(singular_values, 3))

# Оставляем только две главные составляющие.
k = 2
X_approx = U[:, :k] @ np.diag(singular_values[:k]) @ VT[:k, :]
print("\nВосстановленная матрица ранга 2:")
print(np.round(X_approx, 2))
print("\nОшибка восстановления:", np.linalg.norm(X_small - X_approx))

### Мини-эксперимент

Поменяйте `k = 2` на `k = 1`. Сравните ошибку восстановления. Чем меньше компонент мы сохраняем, тем сильнее сжатие, но тем больше информации теряем.

## 1.3. Интуиция PCA

PCA ищет новые взаимно перпендикулярные оси:

- первая компонента объясняет максимальную дисперсию;
- вторая — максимальную оставшуюся дисперсию;
- следующие компоненты продолжают тот же принцип.

Перед PCA признаки почти всегда стандартизируют. Иначе столбец, измеряемый тысячами, может доминировать над столбцом со значениями от 0 до 1 только из-за масштаба.

In [ ]:
n = 300
x1 = np.random.normal(0, 2.0, n)
x2 = 0.85 * x1 + np.random.normal(0, 0.65, n)
x3 = -0.35 * x1 + np.random.normal(0, 0.9, n)
X_3d = pd.DataFrame({"x1": x1, "x2": x2, "x3": x3})

X_scaled = StandardScaler().fit_transform(X_3d)
pca_2 = PCA(n_components=2)
X_2d = pca_2.fit_transform(X_scaled)

print("Доля объяснённой дисперсии:", np.round(pca_2.explained_variance_ratio_, 3))
print("Суммарно сохранено:", round(pca_2.explained_variance_ratio_.sum(), 3))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(X_3d["x1"], X_3d["x2"], alpha=0.65)
axes[0].set_title("Исходные коррелирующие признаки")
axes[0].set_xlabel("x1")
axes[0].set_ylabel("x2")

axes[1].scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.65, color="#7c3aed")
axes[1].set_title("Данные после PCA: 3 признака → 2 компоненты")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
plt.tight_layout()
plt.show()

## 1.4. Сколько компонент оставить

Один из практичных вариантов — указать `PCA(n_components=0.95)`. Тогда алгоритм сам оставит минимальное число компонент, которые вместе объясняют не менее 95% дисперсии.

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

X_scaled = StandardScaler().fit_transform(X)
pca_full = PCA().fit(X_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(9, 5))
plt.plot(range(1, len(cum_var) + 1), cum_var, marker="o", markersize=3)
plt.axhline(0.95, color="crimson", linestyle="--", label="95% дисперсии")
plt.xlabel("Количество компонент")
plt.ylabel("Накопленная объяснённая дисперсия")
plt.title("Выбор числа компонент PCA")
plt.legend()
plt.show()

n_95 = np.argmax(cum_var >= 0.95) + 1
print(f"Исходных признаков: {X.shape[1]}")
print(f"Компонент для сохранения 95% дисперсии: {n_95}")

## 1.5. PCA перед классификацией

Сравним логистическую регрессию на исходных стандартизированных признаках и после PCA. Важно: PCA не использует целевой столбец `y`, поэтому увеличение accuracy не гарантировано. Его задача — компактное представление признаков.

Чтобы не допустить утечку данных, `StandardScaler`, `PCA` и модель помещаем в `Pipeline`: они обучаются только на тренировочной выборке.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])

with_pca = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95)),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])

baseline.fit(X_train, y_train)
with_pca.fit(X_train, y_train)

pred_base = baseline.predict(X_test)
pred_pca = with_pca.predict(X_test)

print("Accuracy без PCA:", round(accuracy_score(y_test, pred_base), 4))
print("Accuracy с PCA:  ", round(accuracy_score(y_test, pred_pca), 4))
print("Компонент оставлено:", with_pca.named_steps["pca"].n_components_)

### Практика 

1. Измените порог сохранённой дисперсии: `0.80`, `0.90`, `0.99`.
2. Для каждого значения запишите число компонент и accuracy.
3. Постройте таблицу результатов.
4. Ответьте: в каком варианте компромисс между сжатием и качеством выглядит лучше?

**Дополнительное задание:** визуализируйте объекты датасета в двух компонентах PCA, раскрасив точки по классу.